# LLM v3 Prompt Mode Evaluation

Compare saved `llm_v3` runs grouped by prompt mode. This mirrors the RL evaluator style, but the comparison dimension here is prompt construction rather than reward function.

In [ ]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

PROJECT_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "machine_learning").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from machine_learning.LLM.llm_v3.analysis_utils import (
    aggregate_run_table,
    collect_run_table,
    draw_metric_panel,
    save_figure,
)

LLM_DIR = PROJECT_ROOT / "machine_learning" / "LLM" / "llm_v3"
ARTIFACT_ROOT = LLM_DIR / "artifacts"
FIGURE_ROOT = LLM_DIR / "prompt_mode_evaluation_figures"


In [ ]:
EVAL_SPLIT = "test"  # "validation" or "test"
MODEL_NAME_FILTER = None  # Example: "gpt-5.4-mini"
FEATURE_SUBSET_FILTER = None  # Example: "full_structural"
SAVE_FIGURES = True

run_table = collect_run_table(ARTIFACT_ROOT, split_name=EVAL_SPLIT)
if MODEL_NAME_FILTER:
    run_table = run_table[run_table["model_name"] == MODEL_NAME_FILTER].copy()
if FEATURE_SUBSET_FILTER:
    run_table = run_table[run_table["feature_subset"] == FEATURE_SUBSET_FILTER].copy()

display(run_table.sort_values("created_at_utc", ascending=False).head(20))


In [ ]:
prompt_mode_summary = aggregate_run_table(run_table, "prompt_mode")
prompt_mode_summary = prompt_mode_summary.rename(columns={"prompt_mode": "method"})
display(prompt_mode_summary)


In [ ]:
panel_fig = draw_metric_panel(
    prompt_mode_summary,
    title=f"Prompt Mode Comparison on {EVAL_SPLIT.title()} Runs",
)
if SAVE_FIGURES:
    save_figure(panel_fig, FIGURE_ROOT, f"prompt_mode_{EVAL_SPLIT}_comparison_panel")
panel_fig.show()
